# 如何并行调用运行接口

`RunnableParallel` 原语本质上是一个字典，其值是运行接口（或可以被强制转换为运行接口的事物，如函数）。

它*并行运行所有值*，并且每个值都使用 RunnableParallel 的*整体输入进行调用*。最终返回值是一个*字典*，包含每个值在其适当键下的结果。

```md     
    Input
      / \
     /   \
 Branch1 Branch2
     \   /
      \ /
      Combine```

In [1]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv("apikey.env")
BASE_URL = 'https://api.deepseek.com'
API_KEY = os.getenv('DEEPSEEK-API-KEY')
deepseek_chat_model = 'deepseek-chat'
if  not API_KEY:
    raise ValueError("WARNING: NOT FOUND OPENAI_API_KEY，PLEASE CHECK .env SETING。")
else:
    print("SECESSFULLY!")
model = ChatDeepSeek(api_key=API_KEY, base_url=BASE_URL, model=deepseek_chat_model)

SECESSFULLY!


- 提示的输入预计是一个包含键 "context" 和 "question" 的映射: 
    - 用户输入问题。
    - 使用检索器获取上下文，并将用户输入传递到 "question" 键下。

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

embedding = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")
vectorstore = FAISS.from_texts(
    ["不是...我真服了，你凭什么觉得我次次都会在原地等你啊？"], embedding=embedding
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# The prompt expects input with keys for "context" and "question"
prompt = ChatPromptTemplate.from_template(template)
model = model

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

async for chunk in retrieval_chain.astream("说话人的情感状态是什么样的？"):
    print(chunk, end="", flush=True)

根据提供的对话内容，说话人的情感状态可以判断为 **不满、失望或愤怒**。  
“不是...我真服了，你凭什么觉得我次次都会在原地等你啊？” 这句话带有明显的反问和抱怨语气，表明说话人对对方总是指望自己无条件等待感到不耐烦和委屈。

- **TIPS**: 请注意，当将 RunnableParallel 与另一个 Runnable **组合时**，我们甚至不需要将字典包装在 RunnableParallel 类中——**类型转换会为我们处理**。以下的写法是等价的：

    - ```py
        {"context": retriever, "question": RunnablePassthrough()}
        ```
    - ```py
        ({"context": retriever, "question": RunnablePassthrough()})
        ```
    - ```py
        RunnableParallel(context=retriever, question=RunnablePassthrough())
        ```

## 使用 itemgetter 作为简写

当与 RunnableParallel 结合使用时，您可以使用 Python 的 itemgetter 作为简写来从映射中提取数据


In [9]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

from operator import itemgetter

embedding = HuggingFaceEmbeddings(model_name="moka-ai/m3e-base")
vectorstore = FAISS.from_texts(
    ["""
“亚里士多德在《尼各马可伦理学》中提出，幸福是生命的终极目标和最高善。
他认为幸福不在于享乐、财富或荣誉，而在于‘合乎德性的灵魂活动’。
这意味着幸福是通过理性引导，持续地践行美德（如勇敢、公正、节制）来实现的。
此外，亚里士多德强调幸福是一种‘活动’，而非一种静止的状态，它需要在一个完整的人生中得以体现。”

"""], embedding=embedding
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
Answer in the following language: {language}
"""

# The prompt expects input with keys for "context" and "question"
prompt = ChatPromptTemplate.from_template(template)
model = model

retrieval_chain = (
    {
        "context": (itemgetter("question") | retriever),
        "question": itemgetter("question"),
        "language": itemgetter("language"),
    }
    | prompt
    | model
    | StrOutputParser()
)

async for chunk in retrieval_chain.astream({"question": "亚里士多德对于幸福的主要观点", "language": "汉语和英语"}):
    print(chunk, end="", flush=True)

**汉语：**  
亚里士多德认为幸福是生命的终极目标和最高善，不在于享乐、财富或荣誉，而在于“合乎德性的灵魂活动”。他强调幸福需通过理性引导、持续践行美德（如勇敢、公正、节制）来实现，并指出幸福是一种动态的“活动”，而非静止状态，需在完整的人生中体现。

**English：**  
Aristotle viewed happiness as the ultimate goal and highest good of life, asserting that it lies not in pleasure, wealth, or honor, but in "virtuous activity of the soul." He emphasized that happiness is achieved through rational guidance and the consistent practice of virtues (such as courage, justice, and temperance), noting that it is an active pursuit rather than a static state, requiring fulfillment over a complete life.

- retriever 是一个 Runnable 对象（其实就是一个封装好的语义搜索器），
当你给它一个字符串问题时，它会自动：

    - 把问题转成向量；

    - 在 FAISS 中搜索相似文档；

    - 返回结果文本（context）。

```md
                    ┌────────────────────────────────────────────┐
                    │            输入 Input Dict                  |
                    │ {"question": "亚里士多德对于幸福的主要观点",  │
                    │  "language": "语言"}                      │
                    └──────────────────────┬─────────────────────┘
                                           │
               ┌───────────────────────────┼──────────────────────────┐
               │                           │                          │
               ▼                           ▼                          ▼
      itemgetter("question")       itemgetter("question")      itemgetter("language")
               │                           │                          │
         ┌─────┴─────┐                     │                          │
         │ retriever │                     │                          │
         └─────┬─────┘                     │                          │
               │                           │                          │
               ▼                           ▼                          ▼
          "context"                   "question"                  "language"
               └────────────────────────────┬────────────────────────┘
                                            ▼
                                  ChatPromptTemplate
                                            ▼
                                          model
                                            ▼
                                  StrOutputParser
                                            ▼
                                    最终生成答案
```

# 并行化步骤
RunnableParallels 使得并行执行多个 Runnables 变得简单，并将这些 Runnables 的输出作为映射返回。

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

joke_chain = ChatPromptTemplate.from_template("tell me a joke about {topic}") | model
poem_chain = (
    ChatPromptTemplate.from_template("write a 2-line poem about {topic}") | model
)

map_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)

map_chain.invoke({"topic": "熊熊🐻"})

{'joke': AIMessage(content='为什么熊熊🐻这么喜欢讲笑话？  \n因为每次它一开口，大家都被“熊”住了！  \n（而且它讲完还会自己拍“爪”叫好！🐾）', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 13, 'total_tokens': 54, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 13}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': 'd3f03293-0975-4148-b20f-e63640a532c9', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--1a01eb67-27a1-4054-8de5-396771405308-0', usage_metadata={'input_tokens': 13, 'output_tokens': 41, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
 'poem': AIMessage(content='熊熊，熊熊，圆滚滚，\n毛茸茸的，暖我心。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 15, 'total_tokens': 30, 'completion_

# 并行性
RunnableParallel 也适用于并行运行独立进程，因为映射中的每个 Runnable 都是并行执行的。

In [ ]:
%%timeit

joke_chain.invoke({"topic": "bear"})

2.26 s ± 309 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [20]:
%%timeit

poem_chain.invoke({"topic": "bear"})

2.27 s ± 124 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
%%timeit

map_chain.invoke({"topic": "bear"})

2.39 s ± 292 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
